# Minimi quadrati: fit polinomiale, aggiornamento QR e regressione a componenti principali

## Perché i minimi quadrati

Il metodo dei minimi quadrati fu formulato indipendentemente da Legendre, che lo pubblicò nel 1805, e da Gauss, che sosteneva di usarlo già da alcuni anni e lo pubblicò nel 1809. L'episodio che ne dimostrò la potenza pratica risale al 1801: il pianetino Cerere era stato osservato per poche settimane e poi perso di vista dietro il Sole; Gauss usò le poche osservazioni disponibili per stimarne l'orbita con il proprio metodo, e quando Cerere ridivenne visibile fu ritrovato esattamente nella posizione prevista dal fit — una delle prime dimostrazioni che un modello stimato da dati rumorosi potesse fare previsioni quantitativamente affidabili.

La motivazione è rimasta la stessa da allora: ogni volta che una grandezza fisica è misurata con rumore e si vuole risalire a un andamento sottostante o calibrare un modello, i minimi quadrati sono lo strumento di base. Il punto delicato, che occupa la maggior parte di questo notebook, è che l'approccio più diretto — le equazioni normali — può essere numericamente fragile quando i predittori sono mal condizionati o ridondanti. Il percorso è organizzato in sei step, tutti sullo stesso problema di partenza:

1. **Step 1 — Il problema e i dati.** Un dataset sintetico di misure rumorose, generato da un polinomio noto, per poter verificare più avanti se i metodi usati recuperano l'andamento vero.
2. **Step 2 — Soluzione via fattorizzazione QR.** Il fit polinomiale risolto tramite QR invece delle equazioni normali, con la derivazione di perché questo evita di elevare al quadrato il condizionamento del problema.
3. **Step 3 — Aggiornamento con una rotazione di Givens.** Come aggiornare una fattorizzazione QR già calcolata quando arriva una nuova osservazione, senza rifattorizzare tutto da zero.
4. **Step 4 — Condizionamento del problema.** L'ill-conditioning strutturale della matrice di Vandermonde al crescere del grado, e il suo effetto quantitativo sulla sensibilità del fit al rumore.
5. **Step 5 — Regressione a componenti principali (PCR) via SVD.** Come gestire un problema con predittori quasi ridondanti, troncando la soluzione alle direzioni più robuste.
6. **Step 6 — Confronto con un solutore di libreria.** Verifica di coerenza tra la soluzione calcolata a mano e quella di `scipy.linalg.lstsq`.

## Step 1 — Il problema e i dati

### Formulazione del problema

Il problema dei **minimi quadrati** nasce quando si dispone di un insieme di osservazioni $(x_i, y_i)$, $i=1,\dots,n$, e si vuole trovare, all'interno di una famiglia di funzioni scelta a priori, quella che meglio approssima i dati nel senso della somma dei quadrati degli scarti. Qui la famiglia scelta è quella dei polinomi di grado $m$:

$$p(x) = c_0 + c_1 x + c_2 x^2 + \dots + c_m x^m,$$

e il problema consiste nel trovare i coefficienti $c_0,\dots,c_m$ che minimizzano

$$\sum_{i=1}^{n} \big(y_i - p(x_i)\big)^2.$$

Il modello statistico sottostante, che useremo per generare i dati di questo notebook, assume che le osservazioni siano una funzione ignota $f$ (qui, per costruzione, un polinomio di grado noto) più un errore di misura:

$$y_i = f(x_i) + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \text{ indipendenti}.$$

Generare i dati con $f$ nota e fissata è una scelta didattica precisa: permette, negli step successivi, di verificare che il fit calcolato via QR (Step 2) recuperi effettivamente l'andamento vero, e non solo che produca una curva visivamente plausibile.

### Il dataset: coefficiente di resistenza aerodinamica in funzione della velocità

I dati simulano una serie di misure del **coefficiente di resistenza aerodinamica** $C_d$ di un profilo, raccolte a diverse velocità $v$ del flusso incidente. In realtà $C_d$ dipende dalla velocità in modo complesso (attraverso il numero di Reynolds, effetti di comprimibilità alle alte velocità, transizione dello strato limite), ma per gli scopi di questo notebook basta un andamento sintetico non banale: useremo un polinomio cubico noto come "verità" $f(v)$, a cui sovrapponiamo un rumore di misura gaussiano di piccola ampiezza, con seed fissato per la riproducibilità.

Importiamo le librerie usate in tutto il notebook: `numpy` per il calcolo vettoriale, `matplotlib.pyplot` per i grafici, `scipy.linalg` per la fattorizzazione QR e la soluzione di sistemi triangolari. Le funzioni riutilizzabili sviluppate in questo notebook — il fit ai minimi quadrati via QR (Step 2 e 4), l'aggiornamento della fattorizzazione con rotazioni di Givens (Step 3), il calcolo del condizionamento del Vandermonde (Step 4) e la costruzione della soluzione troncata via SVD (Step 5) — vivono nel modulo condiviso `src/least_squares.py` e vengono importate da lì.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la
from src.least_squares import qr_lstsq, qr_polyfit, givens_qr_update, vandermonde_conditioning, pcr_solution_path

### Parametri del dataset

Scegliamo $n=40$ misure, con velocità $v$ campionata uniformemente nell'intervallo $[10, 60]\ \mathrm{m/s}$ (ordinata, per comodità di lettura dei grafici) e polinomio generatore di grado 3

$$f(v) = 10^{-6} v^3 - 1.5\times10^{-4} v^2 + 0.01\, v + 0.25,$$

coefficienti scelti per dare un andamento crescente e debolmente curvo su questo intervallo, con rumore $\varepsilon \sim \mathcal{N}(0, 0.008^2)$.

In [ ]:
np.random.seed(42)

n = 40
v = np.sort(np.random.uniform(10, 60, n))

f_true = np.array([1e-6, -1.5e-4, 0.01, 0.25])  # coefficienti di f, grado decrescente (convenzione np.polyval)
eps = np.random.normal(0, 0.008, n)

y = np.polyval(f_true, v) + eps

Visualizziamo i dati grezzi $(v, y)$: solo i punti misurati, rumorosi, senza alcun fit sovrapposto — il confronto con la curva stimata via QR arriva nello Step 2.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(v, y, "o", color="tab:blue")
ax.set_xlabel("v [m/s]")
ax.set_ylabel("$C_d$")
ax.set_title("Coefficiente di resistenza aerodinamica: dati grezzi")
plt.show()

## Step 2 — Soluzione via fattorizzazione QR

### Formulazione matriciale

Costruendo la **matrice di Vandermonde** $A \in \mathbb{R}^{n\times(m+1)}$, con la colonna $j$-esima uguale a $v^{m-j}$, il fit polinomiale di grado $m$ diventa un sistema lineare sovradeterminato

$$Ap \approx y, \qquad p = (c_m, c_{m-1}, \dots, c_0)^T,$$

e minimizzare $\sum_i (y_i - p(v_i))^2$ equivale a minimizzare $\|Ap-y\|_2^2$.

Scegliamo qui **grado $m=3$**: lo stesso grado del polinomio generatore $f(v)$ usato nello Step 1. La scelta non è casuale — permette di verificare, nel confronto numerico più sotto, se il fit recupera effettivamente i coefficienti con cui i dati sono stati generati, e non solo di ottenere una curva visivamente plausibile sopra ai punti.

### Soluzione tramite fattorizzazione QR

Anziché risolvere le equazioni normali $A^TAp=A^Ty$, che richiederebbero di formare $A^TA$ e quindi di elevare al quadrato il numero di condizionamento di $A$ (il legame preciso tra condizionamento ed errore sarà quantificato nello Step 4), conviene fattorizzare $A$ con la **QR economica**

$$A = QR, \qquad Q\in\mathbb{R}^{n\times(m+1)} \text{ a colonne ortonormali}, \quad R\in\mathbb{R}^{(m+1)\times(m+1)} \text{ triangolare superiore}.$$

Scomponiamo $y$ nella sua parte nel range di $Q$ e nella parte ortogonale ad esso: $y = Q(Q^Ty) + \big(y - QQ^Ty\big)$, dove il secondo termine è ortogonale a $\mathrm{range}(Q)$ e quindi anche a $QRp$ per ogni $p$ (essendo $QRp \in \mathrm{range}(Q)$). Per il teorema di Pitagora,

$$\|Ap-y\|_2^2 = \|QRp - y\|_2^2 = \|Rp - Q^Ty\|_2^2 + \|y-QQ^Ty\|_2^2,$$

e il secondo addendo non dipende da $p$: la somma è minima quando il primo si annulla, cioè quando

$$Rp = Q^Ty.$$

Essendo $R$ triangolare superiore (e invertibile, se $A$ ha rango pieno), questo sistema si risolve per sostituzione all'indietro, senza mai formare $A^TA$.

In [ ]:
m = 3
A = np.vander(v, m + 1)
p = qr_lstsq(A, y)

Confrontiamo i coefficienti $p$ appena stimati con quelli del polinomio generatore $f_{\text{true}}$ dello Step 1: essendo il rumore piccolo rispetto alla scala dei dati, ci aspettiamo che il fit li recuperi con uno scarto contenuto, coerente con l'ampiezza del rumore stesso.

In [ ]:
print("coefficienti stimati (QR):", p)
print("coefficienti veri (f_true):", f_true)
print("scarto assoluto:", np.abs(p - f_true))

Valutiamo il polinomio stimato su una griglia fine di velocità e sovrapponiamo la curva ai dati grezzi dello Step 1, per un confronto visivo diretto con il rumore delle misure.

In [ ]:
v_grid = np.linspace(v.min(), v.max(), 300)
y_fit = np.polyval(p, v_grid)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(v, y, "o", color="tab:blue", label="dati")
ax.plot(v_grid, y_fit, "-", color="tab:red", label=f"fit QR (grado {m})")
ax.set_xlabel("v [m/s]")
ax.set_ylabel("$C_d$")
ax.set_title("Fit polinomiale via QR")
ax.legend()
plt.show()

## Step 3 — Aggiornamento della fattorizzazione con una rotazione di Givens

Se arriva una nuova osservazione $(v_k, y_k)$ dopo aver già fattorizzato le prime $k$ righe di $A$, rifare la QR da zero su tutte le $k+1$ righe è calcolo sprecato quando $k$ è già grande: si può invece **aggiornare** la fattorizzazione esistente in $O((m+1)^2)$ operazioni, anziché ricalcolarla da capo in $O(k(m+1)^2)$.

Per mostrarlo, prendiamo solo le prime $k_0 = m+2 = 5$ righe di $A$ (e di $y$) — una in più delle $m+1=4$ colonne, cosicché il problema resti sovradeterminato — e ne calcoliamo la QR economica $A_0 = Q_0R_0$, con termine noto trasformato $z_0 = Q_0^Ty_0 \in \mathbb{R}^4$.

In [ ]:
k0 = m + 2
A0, y0 = A[:k0], y[:k0]
Q0, R0 = la.qr(A0, mode="economic")
z0 = Q0.T @ y0

### Rotazione di Givens

Data la matrice triangolare superiore $R_0$ e una nuova riga $a\in\mathbb{R}^4$ (la riga $k_0$ di $A$) da incorporare, si vuole aggiornare la fattorizzazione di $\begin{pmatrix}R_0\\a^T\end{pmatrix}$ senza ricalcolarla da capo. Lo strumento è la **rotazione di Givens**

$$G(c,s) = \begin{pmatrix}c & s\\ -s & c\end{pmatrix}, \qquad c=\frac{x}{\sqrt{x^2+y^2}}, \quad s=\frac{y}{\sqrt{x^2+y^2}},$$

che, applicata a una coppia di righe $(r_x, r_y)$ con $x$ e $y$ nella stessa posizione $j$, dà

$$G\begin{pmatrix}r_x\\r_y\end{pmatrix} = \begin{pmatrix}c\,r_x+s\,r_y\\-s\,r_x+c\,r_y\end{pmatrix},$$

la cui seconda riga ha uno zero in posizione $j$, perché $-s\,x+c\,y=0$ per costruzione.

L'algoritmo scorre $j=0,\dots,m$: usa come perno l'elemento diagonale $R_0[j,j]$ e la componente $a[j]$ per calcolare $c,s$; la riga $j$ di $R$ diventa $c\,R_0[j,:]+s\,a$, mentre $a$ diventa $-s\,R_0[j,:]+c\,a$ — che ora ha uno zero in posizione $j$, oltre a quelli già annullati ai passi precedenti. Lo stesso $c,s$ aggiorna anche il termine noto trasformato: $z_0[j]$ e la componente scalare $b$ associata alla nuova osservazione seguono la stessa combinazione. Dopo $m+1$ passi, $a$ è completamente annullato e $R_0,z_0$ sono la fattorizzazione aggiornata.

In [ ]:
R_upd, z_upd = givens_qr_update(R0, z0, A[k0].copy(), y[k0])

### Verifica contro la rifattorizzazione da zero

Confrontiamo `R_upd`/`z_upd` con la QR calcolata direttamente su tutte le $k_0+1=6$ righe. Prima del confronto va normalizzato il segno di ciascuna riga (diagonale di $R$ resa positiva): la fattorizzazione QR è unica solo a meno del segno di ciascuna colonna di $Q$ (e della riga corrispondente di $R$), quindi due implementazioni corrette possono differire per segni pur rappresentando la stessa soluzione.

In [ ]:
A_ext, y_ext = A[:k0 + 1], y[:k0 + 1]
Qf, Rf = la.qr(A_ext, mode="economic")
zf = Qf.T @ y_ext

sign_upd = np.sign(np.diag(R_upd))
sign_full = np.sign(np.diag(Rf))
R_upd_n = sign_upd[:, None] * R_upd
Rf_n = sign_full[:, None] * Rf
z_upd_n = sign_upd * z_upd
zf_n = sign_full * zf

diff_R = np.linalg.norm(R_upd_n - Rf_n, ord=np.inf)
diff_z = np.linalg.norm(z_upd_n - zf_n, ord=np.inf)
print("differenza su R (norma infinito):", diff_R)
print("differenza su z = Q^T y (norma infinito):", diff_z)

## Step 4 — Condizionamento del problema

Il **numero di condizionamento** $\kappa(A) = \sigma_{\max}(A)/\sigma_{\min}(A)$ (rapporto tra il più grande e il più piccolo valore singolare, in norma 2) misura quanto una perturbazione relativa dei dati in ingresso può essere amplificata nella soluzione. Per un sistema ai minimi quadrati risolto via QR, la sensibilità della soluzione a una perturbazione del termine noto $y$ è governata da $\kappa(A)$ — non da $\kappa(A)^2$ come accadrebbe risolvendo le equazioni normali $A^TAp=A^Ty$ (il collegamento già anticipato nello Step 2: formare $A^TA$ eleva al quadrato il condizionamento).

Perché il Vandermonde peggiora al crescere del grado: su un intervallo $[v_{\min},v_{\max}]$ non centrato in $[-1,1]$, le colonne $v^0,v^1,\dots,v^m$ diventano, al crescere di $m$, vettori sempre più simili in direzione — dominati dalle potenze più alte valutate sui valori estremi di $v$ — cioè quasi linearmente dipendenti. È un fenomeno di ill-conditioning classico e strutturale della matrice di Vandermonde, non un artefatto costruito ad hoc: lo osserviamo qui direttamente sulla stessa matrice usata negli Step 2-3, facendone crescere il grado.

In [ ]:
degrees = np.arange(2, 13)
conds = vandermonde_conditioning(v, degrees)

Plottiamo $\kappa(A)$ in funzione del grado. Data la crescita attesa su più ordini di grandezza, usiamo una scala semilogaritmica sull'asse verticale.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(degrees, conds, "o-", color="tab:blue")
ax.set_xlabel("grado del polinomio (m)")
ax.set_ylabel("cond(A) (scala log)")
ax.set_title("Condizionamento della matrice di Vandermonde al crescere del grado")
plt.show()

### Sensibilità del fit al rumore

Colleghiamo ora il condizionamento appena calcolato a un effetto concreto: perturbiamo $y$ con un rumore aggiuntivo piccolo e fissato (seed proprio, per isolare questo esperimento da quello già usato nello Step 1), rifacciamo il fit ai due gradi estremi della griglia sopra ($m=2$ e $m=12$) con la stessa tecnica QR dello Step 2 — generalizzata a un grado qualsiasi tramite la funzione `qr_polyfit` — e confrontiamo l'errore relativo sui coefficienti stimati nei due casi.

Ci aspettiamo, in base a $\kappa(A)$: a grado basso (ben condizionato) il fit deve restare quasi identico; a grado alto (mal condizionato, $\kappa(A)$ enorme) lo stesso piccolo rumore deve produrre una variazione dei coefficienti molto più marcata. Eventuali avvisi numerici di scipy/numpy sulla quasi-singolarità della matrice a grado 12 sono attesi e fanno parte del fenomeno mostrato, non un errore da correggere.

In [ ]:
np.random.seed(7)
y_pert = y + np.random.normal(0, 1e-4, n)

Calcoliamo il fit (dati originali e dati perturbati) ai due gradi estremi, e l'errore relativo $\|p_{\text{pert}}-p\|_2/\|p\|_2$ in ciascun caso.

In [ ]:
deg_low, deg_high = int(degrees[0]), int(degrees[-1])

p_low = qr_polyfit(v, y, deg_low)
p_low_pert = qr_polyfit(v, y_pert, deg_low)
p_high = qr_polyfit(v, y, deg_high)
p_high_pert = qr_polyfit(v, y_pert, deg_high)

rel_err_low = np.linalg.norm(p_low_pert - p_low) / np.linalg.norm(p_low)
rel_err_high = np.linalg.norm(p_high_pert - p_high) / np.linalg.norm(p_high)

print(f"grado {deg_low}: cond(A) = {conds[0]:.2e}, errore relativo sui coefficienti = {rel_err_low:.2e}")
print(f"grado {deg_high}: cond(A) = {conds[-1]:.2e}, errore relativo sui coefficienti = {rel_err_high:.2e}")

## Step 5 — Regressione tramite componenti principali (PCR) via SVD

### La soluzione a norma minima e il suo punto debole

Per un sistema $Ax\approx b$ con $A=U\Sigma V^T$ e $r$ valori singolari non nulli (o comunque numericamente significativi), la soluzione a norma minima è

$$x = \sum_{i=1}^{r}\frac{u_i^Tb}{\sigma_i}\,v_i.$$

Se alcuni $\sigma_i$ sono estremamente piccoli — il caso tipico quando due o più colonne di $A$ sono quasi collineari — includerli nella somma significa dividere per un numero quasi nullo: è lo stesso meccanismo di amplificazione del rumore già visto nello Step 4 tramite $\kappa(A)=\sigma_{\max}/\sigma_{\min}$, qui applicato direzione per direzione invece che in aggregato. Il punto delicato è che questa direzione può contribuire pochissimo alla riduzione del residuo di addestramento, pur rendendo la soluzione molto più sensibile al rumore.

L'idea della **regressione a componenti principali (PCR)** è troncare la somma ai primi $k$ termini (i valori singolari più grandi):

$$x_k = \sum_{i=1}^{k}\frac{u_i^Tb}{\sigma_i}\,v_i,$$

accettando un residuo di addestramento leggermente più alto in cambio di una soluzione molto meno sensibile al rumore. Il valore di $k$ si sceglie osservando come il residuo relativo $\|Ax_k-b\|_2/\|b\|_2$ decresce al crescere di $k$.

Questa idea di troncare alle direzioni più significative non è specifica dei sistemi lineari: è la stessa alla base del **teorema di Eckart-Young**, secondo cui, data una matrice $A=U\Sigma V^T$, la matrice di rango $k$ più vicina ad $A$ (in norma di Frobenius o in norma 2) è esattamente la sua SVD troncata $A_k=U_{:,:k}\Sigma_{:k,:k}V_{:k,:}^T$ — nessun'altra matrice di rango $\le k$ approssima $A$ meglio. Qui non stiamo approssimando la matrice $A$ in sé, ma applicando la stessa logica alla soluzione del sistema lineare: si mantiene solo l'informazione associata alle direzioni più robuste (valori singolari grandi) e si scarta quella associata alle direzioni più fragili (valori singolari piccoli, le prime ad amplificare il rumore).

### Un problema con una feature ridondante

Scenario: prevedere la **forza di deportanza** $F_d$ generata da uno spoiler a partire da 4 misure di sensori simulati — velocità locale del vento $s_1$, temperatura dell'aria $s_2$, umidità relativa $s_3$ — più un quarto sensore $s_4$, un **secondo sensore di temperatura ridondante**: quasi un duplicato di $s_2$, con un piccolo offset di calibrazione e un rumore di misura proprio. $F_d$ dipende realmente solo da $s_1,s_2,s_3$: $s_4$ non porta alcuna informazione aggiuntiva, essendo quasi una combinazione lineare di $s_2$, il che rende la matrice di disegno $A_{\text{pcr}}=[s_1\ s_2\ s_3\ s_4]$ numericamente quasi rango-deficiente pur avendo 4 colonne.

In [ ]:
np.random.seed(101)

n_pcr = 25
s1 = np.random.uniform(15, 55, n_pcr)   # velocità locale del vento [m/s]
s2 = np.random.uniform(10, 35, n_pcr)   # temperatura dell'aria [°C]
s3 = np.random.uniform(30, 90, n_pcr)   # umidità relativa [%]
s4 = s2 + 0.8 + np.random.normal(0, 0.05, n_pcr)  # sensore di temperatura ridondante (quasi duplicato di s2)

w_true = np.array([1.2, -0.3, 0.05])  # pesi veri per s1, s2, s3 (s4 non contribuisce: è ridondante)
b_pcr = w_true[0] * s1 + w_true[1] * s2 + w_true[2] * s3 + np.random.normal(0, 0.5, n_pcr)

A_pcr = np.column_stack([s1, s2, s3, s4])

Calcoliamo la SVD di $A_{\text{pcr}}$ e osserviamo i valori singolari: se lo scenario di ridondanza è come descritto, l'ultimo deve risultare nettamente più piccolo degli altri tre.

In [ ]:
U, s, Vt = np.linalg.svd(A_pcr, full_matrices=False)
print("valori singolari:", s)

Costruiamo $x_k$ incrementalmente: a ogni passo aggiungiamo un termine alla somma della formula sopra e registriamo il residuo relativo $\|A_{\text{pcr}}x_k-b_{\text{pcr}}\|_2/\|b_{\text{pcr}}\|_2$.

In [ ]:
n_cols = A_pcr.shape[1]
x_k_list, residuals = pcr_solution_path(A_pcr, b_pcr)
ks = np.arange(1, n_cols + 1)

Plottiamo il residuo relativo in funzione di $k$, in scala semilogaritmica.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(ks, residuals, "o-", color="tab:blue")
ax.set_xlabel("k (numero di componenti usate)")
ax.set_ylabel("residuo relativo $\\|Ax_k-b\\|/\\|b\\|$ (scala log)")
ax.set_title("Residuo della soluzione troncata al crescere di k")
plt.show()

### Scelta di k e confronto di stabilità

Scegliamo il $k$ minimo che porta il residuo sotto una soglia ragionevole (`2e-2`), invece di usare sempre tutte le componenti disponibili. Confrontiamo poi la soluzione troncata scelta con quella completa ($k=4$, che include anche la componente associata al valore singolare più piccolo): se la coppia di sensori quasi ridondanti ($s_2,s_4$) è davvero la causa della quasi-singolarità, i coefficienti stimati per $s_2$ e $s_4$ nella soluzione completa devono differire sensibilmente da quelli della soluzione troncata, a fronte di un residuo che invece resta quasi identico — il segno che l'ultima componente aggiunge instabilità senza un reale guadagno nel fit.

In [ ]:
threshold = 2e-2
k_idx = next(i for i, r in enumerate(residuals) if r < threshold)
k_choice = ks[k_idx]
x_choice = x_k_list[k_idx]
x_full = x_k_list[-1]

print(f"k scelto: {k_choice} (residuo = {residuals[k_idx]:.2e}, soglia = {threshold:.0e})")
print(f"residuo a k pieno ({n_cols}):", residuals[-1])
print("soluzione troncata  x_k:  ", x_choice)
print("soluzione completa  x_full:", x_full)

## Step 6 — Confronto con un solutore di libreria

Come controllo di coerenza, risolviamo lo stesso problema di minimi quadrati dello Step 2 (stessa `A`, stessa `y`, grado $m=3$) con il solutore di libreria `scipy.linalg.lstsq` — che internamente può usare SVD o QR a seconda del driver LAPACK scelto — e confrontiamo il risultato con la soluzione calcolata a mano via QR: a meno di errori numerici trascurabili, devono coincidere.

In [ ]:
p_lib, res_lib, rank_lib, sv_lib = la.lstsq(A, y)

print("soluzione libreria (lstsq):", p_lib)
print("soluzione QR a mano:       ", p)
print("coincidono (np.allclose):", np.allclose(p_lib, p))

## Conclusioni e limiti

Il filo conduttore del notebook è stato un unico problema di minimi quadrati, affrontato con una catena di tecniche numericamente robuste — fattorizzazione QR al posto delle equazioni normali, aggiornamento incrementale con rotazioni di Givens, analisi esplicita del condizionamento, regressione a componenti principali per il caso di predittori ridondanti — e infine verificato contro un solutore di libreria.

Vale la pena essere espliciti su alcuni limiti, per non lasciare l'impressione che le scelte fatte qui siano automaticamente quelle "giuste" in generale:

- La soglia usata per scegliere $k$ nello Step 5 è fissata a mano, a scopo illustrativo. In pratica un valore così (o, analogamente, la forza di regolarizzazione in una ridge regression, tecnica imparentata con la PCR) si sceglierebbe tramite validazione incrociata su dati tenuti da parte, non guardando solo il residuo sui dati di addestramento.
- L'aggiornamento di Givens dello Step 3 gestisce solo l'arrivo di una nuova riga ("updating"). Un sistema di minimi quadrati in streaming reale deve gestire anche la rimozione di osservazioni vecchie ("downdating") ed eventuali cambi del rango numerico della matrice, che richiedono algoritmi più delicati di quello mostrato qui.
- L'ill-conditioning del Vandermonde osservato nello Step 4 è una proprietà strutturale della base monomiale $1,x,x^2,\dots$, non un difetto dei dati. In pratica, per fit polinomiali di grado elevato si preferiscono basi ortogonali (ad esempio i polinomi di Chebyshev), che evitano il problema alla radice invece di limitarsi a conviverci con QR o SVD.